# Nettoyage des données écologiques

Nettoyage de `monitoring_sites.csv` et `orchard_surveys.csv` sans modifier les sources. Seules les erreurs de format certaines et les lignes strictement dupliquées sont corrigées.

In [ ]:
from pathlib import Path
import re
import pandas as pd

CURRENT_DIR = Path.cwd()
ROOT = CURRENT_DIR if (CURRENT_DIR / 'data' / 'ecology').exists() else CURRENT_DIR.parents[1]
SOURCE_DIR = ROOT / 'data' / 'ecology'
OUTPUT_DIR = SOURCE_DIR / 'cleaned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sites_raw = pd.read_csv(SOURCE_DIR / 'monitoring_sites.csv', dtype=str, keep_default_na=False)
surveys_raw = pd.read_csv(SOURCE_DIR / 'orchard_surveys.csv', dtype=str, keep_default_na=False)
print('Sites :', sites_raw.shape)
print('Relevés :', surveys_raw.shape)

## 1. Diagnostic

Contrôle des valeurs vides, doublons et colonnes numériques contenant du texte.

In [ ]:
numeric_columns = [
    'orchard_area_ha', 'durian_flower_density', 'fruit_set_rate_pct',
    'durian_yield_kg', 'other_pollinator_activity',
    'fertilizer_use_index', 'irrigation_level'
]
print('Doublons exacts — sites :', sites_raw.duplicated().sum())
print('Doublons exacts — relevés :', surveys_raw.duplicated().sum())
print('Identifiants dupliqués :', surveys_raw['orchard_survey_id'].duplicated().sum())
display((sites_raw == '').sum().loc[lambda x: x > 0].rename('vides_sites'))
display((surveys_raw == '').sum().loc[lambda x: x > 0].rename('vides_relevés'))
for column in numeric_columns:
    invalid = pd.to_numeric(surveys_raw[column], errors='coerce').isna().sum()
    if invalid:
        print(f'{column}: {invalid} valeur(s) non directement numérique(s)')

## 2. Nettoyage de `monitoring_sites`

Les espaces et la casse des catégories sont harmonisés, puis les types sont convertis. L'altitude inconnue de `KNT-CYP` reste manquante : aucune valeur fiable ne permet de l'imputer.

In [ ]:
def compact_whitespace(value):
    return re.sub(r'\s+', ' ', value).strip()

sites = sites_raw.copy()
text_columns = sites.columns.difference(['latitude', 'longitude', 'elevation_m'])
sites[text_columns] = sites[text_columns].apply(lambda col: col.map(compact_whitespace))
category_columns = ['site_category', 'habitat_category', 'cave_access_category',
                    'orchard_context', 'site_status']
sites[category_columns] = sites[category_columns].apply(lambda col: col.str.casefold())
for column in ['latitude', 'longitude', 'elevation_m']:
    sites[column] = pd.to_numeric(sites[column], errors='coerce')
for column in ['programme_start_date', 'programme_end_date']:
    sites[column] = pd.to_datetime(sites[column].replace('', pd.NA), errors='raise')

## 3. Nettoyage de `orchard_surveys`

Les unités `ha` et `kg` sont retirées, les virgules décimales converties, et les huit proportions comprises entre 0 et 1 sont remises sur l'échelle 0–100. Une seule occurrence des deux doublons exacts est conservée.

In [ ]:
surveys = surveys_raw.copy()
for column in ['orchard_survey_id', 'site_code', 'survey_period', 'agricultural_disturbance_note']:
    surveys[column] = surveys[column].map(compact_whitespace)

def normalize_numeric(value):
    value = re.sub(r'\s*(ha|kg)\s*$', '', value.strip(), flags=re.IGNORECASE)
    return value.replace(',', '.')

for column in numeric_columns:
    surveys[column] = pd.to_numeric(surveys[column].map(normalize_numeric), errors='raise')
proportion_mask = surveys['fruit_set_rate_pct'].between(0, 1, inclusive='left')
surveys.loc[proportion_mask, 'fruit_set_rate_pct'] = (
    surveys.loc[proportion_mask, 'fruit_set_rate_pct'] * 100
).round(1)
surveys = surveys.drop_duplicates(keep='first').copy()
surveys['survey_period'] = pd.to_datetime(surveys['survey_period'], errors='raise')

## 4. Contrôles et export

L'export est bloqué si une contrainte essentielle échoue. Les valeurs extrêmes plausibles sont conservées.

In [ ]:
assert sites['site_code'].is_unique
assert surveys['orchard_survey_id'].is_unique
assert not surveys.duplicated(['site_code', 'survey_period']).any()
assert set(surveys['site_code']).issubset(set(sites['site_code']))
assert sites['latitude'].between(-90, 90).all()
assert sites['longitude'].between(-180, 180).all()
assert surveys['fruit_set_rate_pct'].between(0, 100).all()
assert (surveys[numeric_columns] >= 0).all().all()
assert surveys.groupby('site_code').size().eq(78).all()

sites.to_csv(OUTPUT_DIR / 'monitoring_sites_cleaned.csv', index=False, date_format='%Y-%m-%d', na_rep='')
surveys.to_csv(OUTPUT_DIR / 'orchard_surveys_cleaned.csv', index=False, date_format='%Y-%m-%d', na_rep='')
print(f'{len(sites)} sites et {len(surveys)} relevés uniques exportés.')
print('Tous les contrôles sont passés.')